## Topic 4: CSV Loading

### Concept, Intuition, Why It Exists

- Targets: `fd_master_database.csv`, `fd_dataset_messy.csv` — tabular data where **each row is a complete, independently meaningful record** (one customer's FD, or one customer email).
- The defining decision, opposite of text loading: **one Document PER ROW**, not one Document for the whole file. A search for "Shobha Chopra's FD status" should retrieve her row, not the entire table dumped as one undifferentiated blob.

### Internal Working

1. Open the CSV with a dict-style reader — each row becomes a dict of `{column_name: value}`, using the header row as keys automatically.
2. For each row, join every column into a single text block as `"column: value"` lines — this is what becomes `page_content`.
3. Wrap it via `make_document()`, recording `{"source": file_path, "row": i}` in metadata — the row index lets you trace any Document straight back to its exact position in the source file for debugging.
4. A lazy generator yields one Document per row without holding the whole file in memory; the eager version materializes the full list.

### Real-World Issues, Edge Cases, Debugging, Security

- **`newline=""` is mandatory** when opening CSVs on some platforms — omitting it can cause extra phantom blank rows.
- **Type coercion landmines**: a dict-style reader returns every value as a **string**. Reading the same CSV through a library with type inference *without* forcing string dtype would silently turn a leading-zero account number into an integer, losing the leading zero permanently. The same discipline that protects this in the database layer must apply at ingestion too.
- **PII in page_content**: every row here includes customer name, mobile number, account number — real(-shaped) PII embedded directly into a vector store. This needs the same access control as the source database; "it's just a vector store" is not a security exemption.
- **Row order drift**: if the source CSV gets re-sorted or re-generated between ingestion runs, the row index no longer points to a stable identity — prefer a true primary key in metadata wherever the source has one.
- **Malformed rows**: a row with a stray unescaped comma can silently misalign columns depending on quoting — validate row length against header length before trusting a row.

### Design Decisions & Trade-offs

- One Document per row vs. one Document per N rows (batched): per-row gives maximum retrieval precision at the cost of many small Documents (more embedding calls). Batching trades precision for fewer, cheaper calls — reasonable for very large tables where row-level granularity isn't actually needed.
- Joining columns as `key: value` text vs. a templated sentence: `key: value` is simple, lossless, works for any schema without a template per table. A templated sentence often embeds *better* semantically but requires hand-maintenance and breaks silently when columns are added.

### Alternatives & When To Use Each

- One Document per row — row = independently meaningful record; need precise per-record retrieval.
- Templated natural-language sentence per row — want better embedding-similarity match to natural questions; willing to maintain a template.
- One Document per whole table — table is small and is always queried/returned as a whole, never per-record.
- Plain structured filtering, no embedding at all — query pattern is always exact-match/range-filter, never semantic; a real DB query beats vector search here.

### Common Mistakes & Production Failures

- Reading without forcing string dtype and silently corrupting leading-zero IDs or phone numbers — a bug that won't surface until someone notices a specific customer lookup failing.
- Embedding obvious PII into a vector store with the same access permissions as a public document set.
- Missing the `newline=""` requirement and getting phantom empty rows that inflate document counts and pollute dedup/embedding costs.

### Lead-Level Interview Questions

**Q: Why one Document per row instead of one per file, given text loading made the opposite choice?**
A: The granularity decision always follows "what is the natural, complete retrieval unit for this content?" For keyword files, that unit is the whole file. For a customer record table, that unit is one row — a query about one customer should retrieve exactly that customer's record, not the entire table or a fragment crossing two unrelated customers.

**Q: This CSV contains real PII columns. Walk through what changes between "fine for a notebook demo" and "fine for production."**
A: PII fields shouldn't be embedded in plaintext without justification — consider redaction for fields not needed for semantic search; the vector store needs the same RBAC as the source database; you need audit logging on what was retrieved and by whom; and deleting a customer's row from the source must also delete their corresponding vector — which a naive ingestion pipeline does not handle by default.

**Q: A type-coercion bug silently turned an account number into an integer during ingestion. How would you have caught this before production?**
A: A schema/contract check at ingestion time — assert each expected string column actually came back as a string matching an expected format, rather than trusting the reader's type inference. Validate structurally at the boundary, don't assume the reader did the right thing.

### Hidden Concepts Worth Knowing

- **Dict-style CSV reading vs. a dataframe library**: the dict-style reader is dependency-free and streams row-by-row naturally, good for the generator pattern. A dataframe library is faster for large files and offers richer dtype control, but reads more eagerly by default.
- **Quoting and escaping**: CSV's biggest hidden complexity is fields containing the delimiter itself — correctly quoted CSVs handle this fine by default, but hand-edited or badly-exported CSVs frequently don't, producing silently misaligned columns.

### Revision Summary

> CSV loading turns every row into its own Document via `key: value` text joining, because a row is the natural complete retrieval unit for tabular records. Pin string types for ID-like columns to avoid silent leading-zero corruption. PII in `page_content` means the vector store inherits the source DB's security requirements.

In [1]:
"""
csv_loader.py
--------------
Loads tabular files (fd_dataset_messy.csv, fd_master_database.csv) as ONE
Document per ROW. Every column is folded into page_content as "key: value"
lines, and the row index is preserved in metadata so any later debugging
can trace a Document straight back to its row in the source file.

Design choice: row-level granularity, not whole-file. A customer's FD
record is the natural retrieval unit -- you want a search to return "this
one customer's record", not "the entire 20,000-row table".
"""

import csv
from document import make_document


def lazy_load_csv(file_path: str):
    """Generator version -- reads row by row, never holds the whole file in memory."""
    with open(file_path, encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)          # each row -> {column_name: value}
        for i, row in enumerate(reader):
            content = "\n".join(f"{k}: {v}" for k, v in row.items())
            yield make_document(
                page_content=content,
                metadata={"source": file_path, "row": i},
            )


def load_csv(file_path: str) -> list:
    """Eager version -- use only when the file is small enough to fit in memory."""
    return list(lazy_load_csv(file_path))


if __name__ == "__main__":
    docs = load_csv("../data/fd_master_database.csv")
    print(f"Loaded {len(docs)} rows as Documents")
    print(f"  Document 0 metadata     : {docs[0]['metadata']}")
    print(f"  Document 0 content[:120]: {docs[0]['page_content'][:120]!r}")

Loaded 20 rows as Documents
  Document 0 metadata     : {'source': '../data/fd_master_database.csv', 'row': 0}
  Document 0 content[:120]: 'FD_No: BJ2019FD7717\nCustomer_Name: Shobha Chopra\nMobile_Number: 9686667744\nAccount_Number: 81960013389083\nFD_Amount_INR:'
